In [1]:
import os
import pandas as pd
import textwrap
import logging
import nest_asyncio

from termcolor import colored
from langgraph.prebuilt import ToolNode
from dotenv import load_dotenv
from typing import TypedDict
from langchain_ollama import ChatOllama
from exa_py import Exa
from langchain_community.vectorstores import FAISS
from mcp.server.fastmcp import FastMCP

#nest_asyncio.apply()
DEBUG=True

In [2]:
logger = logging.getLogger("my-logger")
if DEBUG: 
    logger.setLevel(logging.DEBUG)
else:
    logger.setLevel(logging.INFO)


if not logger.hasHandlers():
    console_handler = logging.StreamHandler()
    logger.addHandler(console_handler)

In [3]:
def pprint(obj, color='white'):
    wrapper = textwrap.TextWrapper(width=82)
    wrapped_text = wrapper.wrap(str(obj))
    print(colored("\n".join(wrapped_text), color=color))

In [4]:
#llm = ChatOllama(model = "qwen3:32b", temperature=0.0, top_k=2, top_p=0.9, verbose=True)

In [5]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import asyncio

server_params = StdioServerParameters(
    command="python",
    args=["calculator_server.py"],
    env=None
)

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Available tools:", [tool.name for tool in tools.tools])
            result = await session.call_tool("add", {"a": 5, "b": 3})
            print("Result of add(5, 3):", result.content.text)

# await asyncio.run(run())   
#await run()

In [7]:
loop = asyncio.get_event_loop()

In [ ]:

loop.create_task(run())

In [ ]:
loop = asyncio.get_event_loop()
asyncio.run_coroutine_threadsafe(ru(), loop)

<Future at 0x7fe925b2a690 state=pending>

In [8]:
loop.run_until_complete(asyncio.gather(run()))

RuntimeError: This event loop is already running

In [ ]:
run()

<coroutine object run at 0x7e910ed8a320>

In [14]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "search_web": {
            "url": "http://localhost:8000/search_web",
            "transport": "streamable_http",
        }
}
)

In [15]:
tools = await client.get_tools()

  + Exception Group Traceback (most recent call last):
  |   File "/home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3670, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/tmp/ipykernel_106763/229651158.py", line 1, in <module>
  |     tools = await client.get_tools()
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/langchain_mcp_adapters/client.py", line 142, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/asyncio/tasks.py", line 385, in __wakeup
  |     future.result()
  |   File "/home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
  |     result = coro.send(None)
  |

In [ ]:
from langgraph.prebuilt import create_react_agent
agent = create_react_agent(llm, tools=tools)

TypeError: 'coroutine' object is not iterable

In [12]:
math_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
)

NameError: name 'agent' is not defined

In [5]:
exec(open("server.py").read())

RuntimeError: Already running asyncio in this thread

In [5]:
from mcp import ClientSession, StdioServerParameters
from pathlib import Path
    
server_parameters = StdioServerParameters(
        command="uv",
        args=["run", "python", "server.py"],
        cwd=str(Path.cwd())
    )

In [8]:
%%bash
fastmcp run server.py:mcp --transport=stdio --port 8080 --host 0.0.0.0

bash: line 1: fastmcp: command not found


CalledProcessError: Command 'b'fastmcp run server.py:mcp --transport=stdio --port 8080 --host 0.0.0.0\n'' returned non-zero exit status 127.

In [19]:
# Load .env file
load_dotenv()
api_key = os.getenv("BRAVE_API_KEY")
exa_api_key = os.getenv("EXA_API_KEY", " ")
exa = Exa(api_key=exa_api_key)
os.environ['FIRECRAWL_API_KEY'] = ''
# Default search config
websearch_config = {
    "parameters": {
        "default_num_results": 5,
        "include_domains": []
    }
}

In [20]:
from langchain_community.tools import BraveSearch
search_tool = BraveSearch.from_api_key(api_key=api_key, search_kwargs={"count": 1})
# tools = [search_tool]
# tool_node = ToolNode(tools=tools)

In [21]:
mcp = FastMCP(
    name="web_search",
    version="1.0.0",
    description="web search capbility using Brave Search"
)

@mcp.tool()
def search_web(query: str) -> str:
    """
    Search the web using Brave Search.
    """
    results = search_tool.invoke(query)
    if not results:
        return "No results found."
    
    # Format the results
    formatted_results = "\n".join([f"{i+1}. {result['title']}: {result['link']}" for i, result in enumerate(results)])
    return f"Search results:\n{formatted_results}"

In [26]:
import asyncio
from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.prebuilt import create_react_agent

async def run_agent():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            agent = create_react_agent(model, tools)
            agent_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})
            return agent_response

In [28]:
tools = load_mcp_tools(mcp)
agent = create_react_agent(llm, tools)
agent_response = agent.invoke({"messages": "What are the news about AI?"})

Exception ignored in: <coroutine object run_agent at 0x72b2f08f2df0>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object run_agent at 0x72b2f08f2df0>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'


TypeError: 'coroutine' object is not iterable

In [27]:
result = asyncio.run(run_agent())
print(result)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [21]:
# Constants for web content fetching
MAX_RETRIES = 3
FIRECRAWL_TIMEOUT = 30  # seconds

In [22]:
from typing import Tuple

def format_search_results(search_results): 
    if not search_results.results:
        return "No results found."
    # Format the search results as needed
    markdown_results = '### Search results:\n\n'
    for idx, result in enumerate(search_results.results, 1):
        title = result.title if hasattr(result, 'title') and result.title else "No title"
        url = result.url 
        published_date = f" (Published: {result.published_date})" if hasattr(result, "published_date") and result.published_date else ""
        markdown_results += f"{idx}. [{title}]({url}){published_date}\n"
        if hasattr(result, 'summary') and result.summary:
            markdown_results += f"> **Summary:** {result.summary}\n\n"
        else:
            markdown_results += "\n"

    return markdown_results


async def search_web(query: str, num_results: int = None) -> Tuple[str, list]:
    """Search the web using Exa API and return both formatted results and raw results."""
    try: 
        search_args = {
            "num_results": num_results or websearch_config["parameters"]["default_num_results"]
        }
        search_results = exa.search_and_contents(
            query,
            summary={"query", "Main points and key takeways"}
            **search_args
        )
        formatted_results = format_search_results(search_results)
        return formatted_results, search_results.results
    except Exception as e:
        print(f"Error occurred while searching the web: {e}")
        return "Error occurred", []

In [23]:
import asyncio
import requests
from langchain_core.documents import Document
from typing import List
from langchain_community.document_loaders.firecrawl import FireCrawlLoader

async def get_web_content(url: str) -> List[Document]:
    """Get web content and convert to document list."""
    for attempt in range(MAX_RETRIES):
        try:
            loader = FireCrawlLoader(
                url=url,
                mode="scrape"
            )
            documents = await asyncio.wait_for(loader.aload(), timeout=FIRECRAWL_TIMEOUT)
            if documents and len(documents) > 0:
                return documents 

            # retry if no documents but no exception 
            if attempt < MAX_RETRIES - 1:
                await asyncio.sleep(1)
                continue

        except requests.exceptions.HTTPError as e:
            if "Website Not Supported" in str(e): 
                print(f"Website not supported: {url}")
                content = f"Content from {url} could not be retrieved. Website not supported"
                return [Document(page_content=content, metadata={"source": url, "error": "Website not supported"})]
            else:
                print(f"HTTP error retrieving content from {url}: {str(e)} (attempt {attempt - 1}/{MAX_RETRIES})")
            if attempt < MAX_RETRIES - 1:
                await asyncio.sleep(1)
                continue
            raise

    return []

            

In [24]:
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

[07/27/25 21:28:38] INFO     Use pytorch device_name: cuda:0                             ]8;id=843786;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=127251;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py#211\211]8;;\

                    INFO     Load pretrained SentenceTransformer: all-MiniLM-L6-v2       ]8;id=458197;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=633843;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py#219\219]8;;\

In [25]:
async def create_rag(links: list[str]) -> FAISS:
    try: 
        model_name = os.getenv("MODEL", "all-MiniLM-L6-v2")
        embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
        documents = []
        tasks = [get_web_content(link) for link in links]
        results = await asyncio.gather(*tasks)
        for result in results:
            if result:
                documents.extend(result)
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, 
                                                       chunk_overlap=200, 
                                                       length_function=len,
                                                       is_separator_regex=False)
        split_docs = text_splitter.split_documents(documents)
        vectorstore = FAISS.from_documents(split_docs, embeddings)
        return vectorstore
    except Exception as e:
        print(f"Error creating RAG: {e}")
        raise e
    
async def create_rag_from_documents(documents: List[Document]) -> FAISS:
    """Create a RAG vector store from a list of documents."""
    try:
        model_name = os.getenv("MODEL", "all-MiniLM-L6-v2")
        embeddings = SentenceTransformerEmbeddings(model_name=model_name, chunk_size=64)
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                                       chunk_overlap=200,
                                                       length_function=len,
                                                       is_separator_regex=False)
        split_docs = text_splitter.split_documents(documents)
        vectorstore = FAISS.from_documents(split_docs, embeddings)
        return vectorstore
    except Exception as e:
        print(f"Error creating RAG from documents: {e}")
        raise e
    
async def search_rag(query: str, rag: FAISS) -> List[Document]:
    """Search the RAG vector store for relevant documents."""
    try:
        if not rag:
            raise ValueError("RAG vector store is not initialized.")
        results = rag.similarity_search(query, k=5)
        return results
    except Exception as e:
        print(f"Error searching RAG: {e}")
        raise e


In [26]:
from mcp.server.fastmcp import FastMCP

In [27]:
mcp = FastMCP(
    name="web_search",
    version="1.0.0",
    description="web search capbility using Exa API, Firecrawl API that provides real-time internet search results and use RAG to search for relevant data. Supports both basic and advanced search with filtering options including domain restrictions, text inclusion requirements, and date filtering. Returns formatted results with titles, URLs, publication dates, and content summaries."
)

In [28]:
@mcp.tool()
async def search_web_tool(query: str) -> str:
    logger.info(f"Searching the web for: {query}")
    formatted_results, raw_results = await search_web(query)

    if not raw_results: 
        return "No search results found."
    
    urls = [result.url for result in raw_results if hasattr(result, 'url')]
    if not urls:
        return "No valid URLs found in search results"
    
    vectorstore = await create_rag(urls)
    rag_results = await search_rag(query, vectorstore)

    full_results = f"{formatted_results}\n\n### RAG Results: \n\n"
    full_results += '\n---\n'.join(doc.page_content for doc in rag_results)

    return full_results 
    

In [29]:
@mcp.tool()
async def get_web_content_tool(url: str) -> str:
    try: 
        documents = await asyncio.wait_for(get_web_content(url), timeout=15.0)
        if documents:
            return f"Content from {url}:\n\n" + "\n\n".join(doc.page_content for doc in documents)
        return "Unable to retrieve content from the provided URL."
    except asyncio.TimeoutError:
        return f"Timeout while trying to retrieve content from {url}. Please try again later."
    except Exception as e:
        return f"An error occurred while retrieving content from {url}: {str(e)}" 

In [30]:
query = "latest news on AI" 
formatted_results, raw_results = await search_web(query)

if not raw_results:
    print("No search results found.")


urls = [result.url for result in raw_results if hasattr(result, 'url')]
if not urls:
    print("No valid URLs found in search results.")
    
print(f"Processing {len(urls)} URLs")

# Create RAG
vectorstore = await create_rag(urls)
rag_results = await search_rag(query, vectorstore)

# Format results
print("\n=== Search Results ===")
print(formatted_results)

print("\n=== RAG Results ===")
for doc in rag_results:
    print(f"\n---\n{doc.page_content}")

Error occurred while searching the web: unsupported operand type(s) for ** or pow(): 'set' and 'dict'
No search results found.
No valid URLs found in search results.
Processing 0 URLs


[07/27/25 21:28:40] INFO     Use pytorch device_name: cuda:0                             ]8;id=800968;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=766257;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py#211\211]8;;\

                    INFO     Load pretrained SentenceTransformer: all-MiniLM-L6-v2       ]8;id=832500;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=547695;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py#219\219]8;;\

[07/27/25 21:28:41] INFO     Loading faiss.                                                           ]8;id=99996;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/faiss/loader.py\loader.py]8;;\:]8;id=578830;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/faiss/loader.py#133\133]8;;\

                    INFO     Successfully loaded faiss.                                               ]8;id=356491;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/faiss/loader.py\loader.py]8;;\:]8;id=467515;file:///home/bmartins/anaconda3/envs/agentic_patterns/lib/python3.12/site-packages/faiss/loader.py#135\135]8;;\

Error creating RAG: list index out of range


IndexError: list index out of range